# 06 — Selección de características y comparación de familias

## Objetivo

Comparar familias de características con el **mismo protocolo** de validación
cruzada y la selección ajustada dentro de cada fold. La pregunta no es solo
“qué modelo gana”, sino **qué información usa**: textura pulmonar, HOG, forma
de la máscara o contexto/fondo.

## 0. Configuración

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
DATA, V2, REP = ROOT / "data", ROOT / "data" / "v2", ROOT / "reports"
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

## 1. Diseño de selección

In [ ]:
from src.modeling import FEATURE_SETS, K_BY_SET, columns_for_set
from src.features import get_feature_groups

feat = pd.read_csv(DATA / "features_tbx11k_v2.csv", nrows=5)
groups = get_feature_groups(feat)
pd.DataFrame([
    {"conjunto": k, "familias": ", ".join(v), "k": K_BY_SET[k]}
    for k, v in FEATURE_SETS.items()
])

La selección usa **ANOVA F** (`f_classif`), no información mutua. Es
determinista, reproducible y no depende de un muestreo estocástico. El
selector se ajusta **solo con el train de cada fold**; nunca ve el test.

## 2. Top features de textura (ANOVA F global, solo para inspección)

In [ ]:
from sklearn.feature_selection import f_classif
feat_full = pd.read_csv(DATA / "features_tbx11k_v2.csv")
y = (feat_full["class_name"] == "tb").astype(int).values
cols = columns_for_set(feat_full, "texture")
F, p = f_classif(feat_full[cols].values, y)
top = pd.DataFrame({"feature": cols, "F": F, "p": p}).sort_values("F", ascending=False).head(20)
top

## 3. Comparación de familias y modelos

In [ ]:
summary = pd.read_csv(V2 / "summary_models.csv")
summary.round(3)

In [ ]:
display(Image(filename=str(REP / "v2" / "comparacion_familias.png")))

## 4. Lectura de los resultados

In [ ]:
pivot = summary.pivot_table(index="feature_set", values=["balacc_youden", "ap_youden"], aggfunc="max")
pivot.round(3).sort_values("balacc_youden", ascending=False)

**Interpretación:** si `context` o `shape` alcanzan un rendimiento alto,
parte del problema es separable sin ver el pulmón; si `texture` es el mejor
conjunto pulmonar, la señal es compatible con tejido. La comparación no
prueba causalidad, pero acota el atajo.

## Conclusiones

- La selección dentro del fold evita fuga de información.
- `texture` es el conjunto principal; `hog`, `shape` y `context` son
  controles explícitos.
- Los resultados de esta tabla alimentan las decisiones del notebook 08 y la
  discusión del 09.

**Siguiente etapa:** `07_Desbalanceo.ipynb`.